In [1]:
import pandas as pd
df = pd.read_excel("data/Online Retail.xlsx")
print(df.head())
print(df.info())
print(df.isnull().sum())


  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----

We can see that there is an issue with the customerID missing values (around 33% of the all customers), for the purpose of analysis we will leave these unknown customers with understanding that those are unregistered users
The missing values in the description category will be removed since they contribute to a small percantage of total data and it will be impossible to analyze the products if we don't know what they are

In [2]:
# CustomerID as nullable int
# Replace missing CustomerID with 0
df["CustomerID"] = df["CustomerID"].fillna(0).astype(int)
df["CustomerID"] = df["CustomerID"].astype("Int64")
df["IsKnownCustomer"] = df["CustomerID"] != 0

# Clean descriptions
df = df.dropna(subset=["Description"])
df["Description"] = df["Description"].str.strip().str.lower()

# Clean country
df["Country"] = df["Country"].str.strip()

# Ensure datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Time features
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["MonthName"] = df["InvoiceDate"].dt.month_name()   # <-- added line
df["Day"] = df["InvoiceDate"].dt.day
df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name().astype(str)

# generate ids for all dates and sales
df["sale_id"] = range(1, len(df) + 1)
df["DateID"] = range(1, len(df) + 1)

# calculate the total price for each sale
df["total_price"] = df["Quantity"] * df["UnitPrice"]

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
Index: 540455 entries, 0 to 541908
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   InvoiceNo        540455 non-null  object        
 1   StockCode        540455 non-null  object        
 2   Description      540454 non-null  object        
 3   Quantity         540455 non-null  int64         
 4   InvoiceDate      540455 non-null  datetime64[us]
 5   UnitPrice        540455 non-null  float64       
 6   CustomerID       540455 non-null  Int64         
 7   Country          540455 non-null  str           
 8   IsKnownCustomer  540455 non-null  boolean       
 9   Year             540455 non-null  int32         
 10  Month            540455 non-null  int32         
 11  MonthName        540455 non-null  str           
 12  Day              540455 non-null  int32         
 13  Hour             540455 non-null  int32         
 14  DayOfWeek        540455 non-null  st

In [7]:
import pandas as pd

# Load the cleaned CSV
df = pd.read_csv("data/online_retail_cleaned.csv", parse_dates=['InvoiceDate'])

# ---- Recreate missing time features ----
df["Year"] = df["InvoiceDate"].dt.year.astype(int)
df["Month"] = df["InvoiceDate"].dt.month.astype(int)
df["MonthName"] = df["InvoiceDate"].dt.month_name().astype(str)
df["Day"] = df["InvoiceDate"].dt.day.astype(int)
df["Hour"] = df["InvoiceDate"].dt.hour.astype(int)
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name().astype(str)

# ---- Customer table ----
customer_df = df[['CustomerID', 'Country', 'IsKnownCustomer']].drop_duplicates()
customer_df.loc[customer_df['CustomerID'] == 0, ['Country','IsKnownCustomer']] = ['Unknown', False]

# Force datatypes
customer_df['CustomerID'] = customer_df['CustomerID'].astype(int)
customer_df['Country'] = customer_df['Country'].astype(str)
customer_df['IsKnownCustomer'] = customer_df['IsKnownCustomer'].astype(bool)

customer_df.to_csv("data/customers.csv", index=False)

# ---- Product table ----
product_df = df[['StockCode', 'UnitPrice', 'Description']].drop_duplicates(subset=['StockCode'])

# Force datatypes
product_df['StockCode'] = product_df['StockCode'].astype(str)
product_df['UnitPrice'] = product_df['UnitPrice'].astype(float)
product_df['Description'] = product_df['Description'].astype(str)

product_df.to_csv("data/products.csv", index=False)

# ---- Date table ----
date_df = df[['DateID', 'InvoiceDate', 'Year', 'Month', 'MonthName', 'Day', 'Hour', 'DayOfWeek']].drop_duplicates(subset=['DateID'])
date_df.rename(columns={
    'InvoiceDate': 'date_stamp',
    'DayOfWeek': 'day_of_week',
    'MonthName': 'month_name'
}, inplace=True)

# Force datatypes
date_df['DateID'] = date_df['DateID'].astype(int)
date_df['date_stamp'] = date_df['date_stamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
date_df['Year'] = date_df['Year'].astype(int)
date_df['Month'] = date_df['Month'].astype(int)
date_df['month_name'] = date_df['month_name'].astype(str)
date_df['Day'] = date_df['Day'].astype(int)
date_df['Hour'] = date_df['Hour'].astype(int)
date_df['day_of_week'] = date_df['day_of_week'].astype(str)

date_df.to_csv("data/dates.csv", index=False)

# ---- Sale table ----
sale_df = df[['sale_id', 'InvoiceNo', 'StockCode', 'DateID', 'CustomerID', 'Quantity', 'total_price']]
sale_df.rename(columns={
    'InvoiceNo': 'invoice_no',
    'StockCode': 'stock_code',
    'Quantity': 'quantity'
}, inplace=True)

# Force datatypes
sale_df['sale_id'] = sale_df['sale_id'].astype(int)
sale_df['invoice_no'] = sale_df['invoice_no'].astype(str)
sale_df['stock_code'] = sale_df['stock_code'].astype(str)
sale_df['DateID'] = sale_df['DateID'].astype(int)
sale_df['CustomerID'] = sale_df['CustomerID'].astype(int)
sale_df['quantity'] = sale_df['quantity'].astype(int)
sale_df['total_price'] = sale_df['total_price'].astype(float)

sale_df.to_csv("data/sales.csv", index=False)

print("All CSVs prepared for Tableau/PostgreSQL with explicit datatypes.")

All CSVs prepared for Tableau/PostgreSQL with explicit datatypes.


In [8]:
print(sale_df[sale_df['total_price'] < 0].head())

     sale_id invoice_no stock_code  DateID  CustomerID  quantity  total_price
141      142    C536379          D     142       14527        -1       -27.50
154      155    C536383     35004C     155       15311        -1        -4.65
235      236    C536391      22556     236       17548       -12       -19.80
236      237    C536391      21984     237       17548       -24        -6.96
237      238    C536391      21983     238       17548       -24        -6.96


Number of rows with CustomerID = 0: 1
